# Create a Hash-Sorted STAC GeoParquet File

This tutorial shows how to compute `hash:hash` values for STAC Items and write those Items to a hash-sorted GeoParquet file.

The result is a small GeoParquet file that keeps nearby Items in space and time closer together on disk. Query engines can still filter on normal STAC fields, but the physical ordering gives them a better chance of skipping irrelevant data.


## Setup

Create a small project for the tutorial with `uv`:

```sh
uv init stac-hash-geoparquet
cd stac-hash-geoparquet
uv add stac-hash pyarrow shapely duckdb ipykernel
uv run python -m ipykernel install --user --name stac-hash-geoparquet
```

Then open this notebook using the `stac-hash-geoparquet` kernel.

This tutorial keeps the input data intentionally small and local. A real dataset is better for benchmarking, but a minimal dataset is easier for learning the publishing workflow because it has no network dependency, credentials, or large downloads.


## What You Will Build

You will create:

- a small set of STAC-like Item records
- a `stac-hash` hasher for the dataset extent
- a `hash:hash` value for each Item
- a GeoParquet file sorted by `hash:hash`
- a quick DuckDB validation query


## Start With STAC Items

In a real pipeline, these records might come from a STAC API search, local Item JSON files, or an existing metadata table. For the tutorial, start with a few minimal STAC-like Items.


In [ ]:
from datetime import UTC, datetime

items = [
    {
        "id": "item-001",
        "collection": "example-scenes",
        "bbox": [-105.2, 39.8, -105.0, 40.0],
        "properties": {
            "datetime": "2026-01-15T18:00:00Z",
            "eo:cloud_cover": 8.0,
        },
        "assets": {
            "data": {
                "href": "s3://example-bucket/item-001.tif",
                "type": "image/tiff; application=geotiff",
            }
        },
    },
    {
        "id": "item-002",
        "collection": "example-scenes",
        "bbox": [-104.9, 40.0, -104.7, 40.2],
        "properties": {
            "datetime": "2026-01-16T18:00:00Z",
            "eo:cloud_cover": 12.0,
        },
        "assets": {
            "data": {
                "href": "s3://example-bucket/item-002.tif",
                "type": "image/tiff; application=geotiff",
            }
        },
    },
    {
        "id": "item-003",
        "collection": "example-scenes",
        "bbox": [-106.1, 38.9, -105.9, 39.1],
        "properties": {
            "datetime": "2026-02-03T18:00:00Z",
            "eo:cloud_cover": 3.0,
        },
        "assets": {
            "data": {
                "href": "s3://example-bucket/item-003.tif",
                "type": "image/tiff; application=geotiff",
            }
        },
    },
]

items[0]


## Choose Hash Extents

Every hash is computed relative to a spatial and temporal extent. The same Item will produce different hashes if you change those extents, so choose values that match the dataset or partition you are writing.

This tutorial uses a regional yearly partition. A smaller spatial extent spends the same number of bits over a smaller area, which gives finer spatial precision.


In [ ]:
spatial_extent = [-110.0, 37.0, -102.0, 42.0]
temporal_start = datetime(2026, 1, 1, tzinfo=UTC)
temporal_end = datetime(2027, 1, 1, tzinfo=UTC)


## Create The Hasher

Pass the temporal extent and bounding box to `Hasher`. Inputs outside the configured extent raise `ValueError`, which prevents writing hash values that cannot be interpreted correctly by the Collection metadata.


In [ ]:
from stac_hash import Hasher

hasher = Hasher(
    temporal_start,
    temporal_end,
    bbox=tuple(spatial_extent),
)


## Compute `hash:hash`

For each Item, parse the datetime and use the center of the bounding box as the representative point.


In [ ]:
def parse_datetime(value: str) -> datetime:
    return datetime.fromisoformat(value.replace("Z", "+00:00")).astimezone(UTC)


def bbox_center(bbox: list[float]) -> tuple[float, float]:
    west, south, east, north = bbox
    return ((west + east) / 2, (south + north) / 2)


for item in items:
    item_datetime = parse_datetime(item["properties"]["datetime"])
    longitude, latitude = bbox_center(item["bbox"])
    item["properties"]["hash:hash"] = hasher.hash_encoded(
        item_datetime,
        longitude,
        latitude,
        "base16",
    )

[(item["id"], item["properties"]["hash:hash"]) for item in items]


This tutorial uses `base16` because it is safe to store in JSON and still sorts consistently as a fixed-width string.


## Add Extension Metadata

The Item-level hash is only meaningful with the Collection-level extents used to compute it. Store the extension URL and the hash extents on the Collection or partition-level Collection.


In [ ]:
collection = {
    "type": "Collection",
    "stac_version": "1.1.0",
    "id": "example-scenes-2026",
    "stac_extensions": [
        "https://stac-extensions.github.io/hash/v0.1.0/schema.json"
    ],
    "hash:spatial_extent": spatial_extent,
    "hash:temporal_extent": [
        temporal_start.isoformat(timespec="seconds").replace("+00:00", "Z"),
        temporal_end.isoformat(timespec="seconds").replace("+00:00", "Z"),
    ],
}

collection


The first version of the extension defaults fixed fields such as `hash:algorithm`, `hash:dtype`, and `hash:temporal_precision`, so they do not need to be repeated unless you want them to be explicit.


## Convert Items To Rows

GeoParquet stores tabular rows. The example below keeps a few common STAC fields, writes the Item geometry as WKB, and stores the assets object as JSON text.


In [ ]:
import json

from shapely.geometry import Point
from shapely.wkb import dumps as dump_wkb


def item_row(item: dict) -> dict:
    longitude, latitude = bbox_center(item["bbox"])
    return {
        "id": item["id"],
        "collection": item["collection"],
        "datetime": parse_datetime(item["properties"]["datetime"]),
        "bbox_xmin": item["bbox"][0],
        "bbox_ymin": item["bbox"][1],
        "bbox_xmax": item["bbox"][2],
        "bbox_ymax": item["bbox"][3],
        "eo:cloud_cover": item["properties"].get("eo:cloud_cover"),
        "hash:hash": item["properties"]["hash:hash"],
        "geometry": dump_wkb(Point(longitude, latitude)),
        "assets": json.dumps(item["assets"], sort_keys=True),
    }


rows = [item_row(item) for item in items]
rows[0]


## Write GeoParquet

Create an Arrow table, attach minimal GeoParquet metadata, sort by `hash:hash`, and write the file.


In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

geo_metadata = {
    "version": "1.1.0",
    "primary_column": "geometry",
    "columns": {
        "geometry": {
            "encoding": "WKB",
            "geometry_types": ["Point"],
            "crs": None,
            "edges": "planar",
            "bbox": spatial_extent,
        }
    },
}

table = pa.Table.from_pylist(rows)
table = table.replace_schema_metadata(
    {
        **(table.schema.metadata or {}),
        b"geo": json.dumps(geo_metadata).encode(),
    }
)
table = table.sort_by([("hash:hash", "ascending")])

pq.write_table(
    table,
    "example-scenes-2026.parquet",
    row_group_size=100_000,
)

table.schema


For small tutorial data, row group size does not matter much. For real datasets, row group sizing affects how much data a query engine can skip. A common starting point is to target row groups large enough for efficient reads but small enough to preserve useful statistics.


## Validate The File

Use DuckDB to confirm the file has the expected rows and hash column.


In [ ]:
import duckdb

con = duckdb.connect()

summary = con.sql(
    """
    SELECT
      count(*) AS rows,
      min("hash:hash") AS first_hash,
      max("hash:hash") AS last_hash
    FROM read_parquet('example-scenes-2026.parquet')
    """
).fetchone()

summary


Check that the rows are sorted by hash.


In [ ]:
hashes = [
    row[0]
    for row in con.sql(
        """
        SELECT "hash:hash"
        FROM read_parquet('example-scenes-2026.parquet')
        """
    ).fetchall()
]

is_sorted = hashes == sorted(hashes)

assert is_sorted
is_sorted


## What To Publish With The File

When you publish the GeoParquet file, also publish the Collection metadata that defines the hash extents. Consumers need those extents to understand how `hash:hash` was computed.

At minimum, include:

```json
{
  "stac_extensions": [
    "https://stac-extensions.github.io/hash/v0.1.0/schema.json"
  ],
  "hash:spatial_extent": [-110.0, 37.0, -102.0, 42.0],
  "hash:temporal_extent": [
    "2026-01-01T00:00:00Z",
    "2027-01-01T00:00:00Z"
  ]
}
```


## Next Steps

The next tutorial shows how to query this layout with DuckDB using normal STAC-style filters. The key idea is that queries still filter on familiar fields like collection, datetime, and bounding box. The hash helps because it changed how records were ordered when the GeoParquet file was written.
